# Phase 4 — RB Model: Tuned XGBoost with Walk-Forward Optuna Search

Trained only on RB's Phase 3-selected 4 features: `scarcity_z`, `vorp_delta_yoy`, `age`, `draft_pick_inverse`. Built `XGBRegressor`-first from the start (not `xgb.train()`/`DMatrix`), matching the interface QB's notebook (`06a_model_qb.ipynb`) was converted to, so the two positions stay consistent going forward. Testing whether tuning meaningfully closes the gap Phase 3's untuned model left against the naive baseline (0.9% MAE, +0.008 Spearman — the smallest lift of any position).

In [1]:
from datetime import datetime

print(f"Results as of {datetime.now().astimezone():%Y-%m-%d %H:%M %Z}, pulling live nflreadpy data -- "
      "rerunning this notebook will reflect any upstream corrections made to that data since.")

Results as of 2026-09-16 13:35 Central Daylight Time, pulling live nflreadpy data -- rerunning this notebook will reflect any upstream corrections made to that data since.


## Setup: load `features_df` and build RB's target rows

Only the 4 already-selected features are built here — not the full Phase 3 pipeline. RB's feature set doesn't include any of the receiving-usage trio, the OL run-blocking proxy, or any of the EPA columns (RFECV eliminated all of them for RB — see `05_feature_selection.ipynb`), so no play-by-play or team-stats pull is needed at all for this notebook.

In [2]:
import sys
from pathlib import Path

import nflreadpy as nfl
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FEATURES = ["scarcity_z", "vorp_delta_yoy", "age", "draft_pick_inverse"]

vorp_labels = pd.read_parquet(REPO_ROOT / "data/processed/vorp_labels.parquet")
rb = vorp_labels[vorp_labels["position"] == "RB"][
    ["season", "player_id", "player_display_name", "vorp", "vorp_next"]
].copy()

# scarcity_z (within-position, within-season standardization -- same recipe as Phase 3)
season_position_stats = (
    vorp_labels.groupby(["season", "position"])["vorp"]
    .agg(position_mean_vorp="mean", position_std_vorp_that_season="std")
    .reset_index()
)
rb_stats = season_position_stats[season_position_stats["position"] == "RB"]
rb = rb.merge(rb_stats[["season", "position_mean_vorp", "position_std_vorp_that_season"]], on="season", how="left")
rb["scarcity_z"] = (rb["vorp"] - rb["position_mean_vorp"]) / rb["position_std_vorp_that_season"]
rb = rb.drop(columns=["position_mean_vorp", "position_std_vorp_that_season"])

# vorp_delta_yoy
prior = rb[["player_id", "season", "vorp"]].copy()
prior["season"] = prior["season"] + 1
prior = prior.rename(columns={"vorp": "vorp_last_season"})
rb = rb.merge(prior, on=["player_id", "season"], how="left")
rb["vorp_delta_yoy"] = rb["vorp"] - rb["vorp_last_season"]
rb = rb.drop(columns=["vorp_last_season"])

# age, draft_pick_inverse
players = nfl.load_players().to_pandas()
rb = rb.merge(players[["gsis_id", "birth_date", "draft_pick"]], left_on="player_id", right_on="gsis_id", how="left")
rb["birth_date"] = pd.to_datetime(rb["birth_date"])
season_start = pd.to_datetime(rb["season"].astype(str) + "-09-01")
rb["age"] = (season_start - rb["birth_date"]).dt.days / 365.25
rb["draft_pick_inverse"] = 1 / rb["draft_pick"]
rb = rb.drop(columns=["gsis_id", "birth_date", "draft_pick"])

# Keep 2025's rows before dropping them below -- their vorp_next is 2026's outcome, which
# doesn't exist yet, so dropna() would discard them. Needed later for the live 2026 inference
# section (added below, closing out a gap QB/WR/TE's notebooks already had).
rb_2025_features = rb[rb["season"] == 2025].copy()

rb = rb.dropna(subset=["vorp_next"]).reset_index(drop=True)
print(f"RB rows with a usable label: {len(rb)}, seasons {rb['season'].min()}-{rb['season'].max()}")
print(rb[FEATURES].isna().mean().rename("null_rate"))

C:\Users\viraj\OneDrive\Desktop\ML Trial\ML-Test\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RB rows with a usable label: 1823, seasons 2008-2024
scarcity_z            0.000000
vorp_delta_yoy        0.297312
age                   0.000000
draft_pick_inverse    0.277564
Name: null_rate, dtype: float64


## Walk-forward folds (same structure as `05_feature_selection.ipynb` / `06a_model_qb.ipynb`)

In [3]:
MIN_TRAIN_SEASONS = 9


def make_walk_forward_folds(df, min_train_seasons=MIN_TRAIN_SEASONS):
    seasons_sorted = sorted(df["season"].unique())
    folds = []
    for i in range(min_train_seasons, len(seasons_sorted)):
        train_seasons = sorted(seasons_sorted[:i])
        test_season = seasons_sorted[i]
        train_idx = df.index[df["season"].isin(train_seasons)].to_numpy()
        test_idx = df.index[df["season"] == test_season].to_numpy()
        if len(train_idx) > 0 and len(test_idx) > 0:
            folds.append((train_idx, test_idx, test_season, train_seasons))
    return folds


folds = make_walk_forward_folds(rb)
print(f"{len(folds)} walk-forward folds, test seasons: {[f[2] for f in folds]}")

8 walk-forward folds, test seasons: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


## Monotonic constraints: checked per feature, not assumed

Each of RB's 4 features checked individually for whether it has an unambiguous, confound-free "more is better/worse" relationship with next-season value:

| Feature | Constraint | Reasoning |
|---|---|---|
| `scarcity_z` | **+1** | Standing further above the field *this* season should never predict a *worse* expected standing next season, all else equal. Unambiguous — same reasoning as QB. |
| `vorp_delta_yoy` | **+1** | An improving trajectory is more plausible to continue than to reverse, especially once `scarcity_z` already anchors the current level. Same reasoning as QB. |
| `age` | **0 (none)** | Explicitly NOT monotonic — and more pronounced for RB than any other position: Phase 3's own feature-importance write-up already named RB's "shelf-life cliff" (production rises then falls hard, with the whole receiving-usage trio getting eliminated once `age` and `scarcity_z` are known). Forcing a single direction here would contradict the real shape of that curve. |
| `draft_pick_inverse` | **0 (none)** | Same real ambiguity as QB: a fixed pre-career proxy for opportunity/organizational investment, not a performance metric. Conditional on `scarcity_z`/`vorp_delta_yoy` already carrying realized performance, it isn't clean enough to force a direction — could be picking up sunk-cost bias as easily as true signal. Left unconstrained rather than guessed. |

**2 of 4 features get a constraint** (`scarcity_z`, `vorp_delta_yoy`, both `+1`); `age` and `draft_pick_inverse` are left free. Same deliberately-conservative rule as QB — a wrong constraint forces an incorrect shape, which is worse than no constraint at all.

In [4]:
MONOTONE_CONSTRAINTS = (1, 1, 0, 0)  # matches FEATURES order exactly
print(dict(zip(FEATURES, MONOTONE_CONSTRAINTS)))

{'scarcity_z': 1, 'vorp_delta_yoy': 1, 'age': 0, 'draft_pick_inverse': 0}


## Per-fold Optuna search (TPE sampler, pruning enabled)

For each outer walk-forward fold, the search space is `max_depth` (2-8), `min_child_weight` (1-10), `reg_lambda` (log-scale 0.1-10), `learning_rate` (log-scale 0.01-0.3), `subsample` (0.6-1.0). `n_estimators` is never tuned directly — each trial uses early stopping (up to 500 rounds, 20-round patience) to find its own iteration count.

`max_depth`'s range was widened from an earlier 2-4 band to 2-8, matching the same change made to `06a_model_qb.ipynb` after `05b_qb_feature_experiment.ipynb` found the shallow-tree cap was artificial for QB — checked here for RB too rather than assuming QB's result carries over.

**The search never touches the outer fold's held-out test season.** Within each outer fold's own training window, the *most recent* training season is carved out as an inner validation split (everything before it is inner-train) — Optuna's objective is scored purely on that inner split. Real mid-training pruning is wired in via a custom `xgboost.callback.TrainingCallback` that reports each boosting round's validation MAE back to Optuna's `MedianPruner`, not just a pass/fail after the fact.

Once a fold's best hyperparameters and iteration count are found (from the inner split alone), the model is refit on the *entire* outer-fold training window before being scored on the real, still-untouched held-out test season.

**Interface**: this notebook is `XGBRegressor`-first from the start — `XGBRegressor(..., callbacks=[OptunaPruningCallback(trial)]).fit(X_train, y_train, eval_set=[(X_valid, y_valid)])`, not `xgb.train()`/`DMatrix`. `XGBRegressor`'s eval sets are auto-named `validation_0`, `validation_1`, ..., so the pruning callback reads `evals_log["validation_0"]` rather than a caller-chosen name.

In [5]:
N_TRIALS = 40


class OptunaPruningCallback(xgb.callback.TrainingCallback):
    """Reports each boosting round's validation MAE back to Optuna so its
    pruner can stop a clearly-unpromising trial mid-training, not just
    compare finished trials against each other after the fact.

    Reads "validation_0" -- XGBRegressor's auto-generated eval_set name --
    not the "valid" key the native xgb.train(evals=[...]) API would use."""

    def __init__(self, trial):
        self.trial = trial

    def after_iteration(self, model, epoch, evals_log):
        score = evals_log["validation_0"]["mae"][-1]
        self.trial.report(score, step=epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()
        return False


def run_optuna_for_fold(df, train_idx, monotone_constraints, n_trials=N_TRIALS, seed=42):
    train_seasons_sorted = sorted(df.loc[train_idx, "season"].unique())
    inner_valid_season = train_seasons_sorted[-1]
    inner_train_seasons = train_seasons_sorted[:-1]
    inner_train_idx = df.index[df["season"].isin(inner_train_seasons) & df.index.isin(train_idx)]
    inner_valid_idx = df.index[(df["season"] == inner_valid_season) & df.index.isin(train_idx)]

    X_train = df.loc[inner_train_idx, FEATURES]
    y_train = df.loc[inner_train_idx, "vorp_next"]
    X_valid = df.loc[inner_valid_idx, FEATURES]
    y_valid = df.loc[inner_valid_idx, "vorp_next"]

    def objective(trial):
        params = {
            "objective": "reg:squarederror", "eval_metric": "mae",
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10, log=True),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "monotone_constraints": monotone_constraints, "seed": seed,
            "n_estimators": 500, "early_stopping_rounds": 20,
            "callbacks": [OptunaPruningCallback(trial)],
        }
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
        trial.set_user_attr("best_iteration", model.best_iteration)
        return model.best_score

    study = optuna.create_study(
        direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study


def fold_gain_share(model):
    """Gain importance for one fold's refit booster, normalized to shares
    over FEATURES (0 for any feature the booster never split on) -- same
    definition Phase 3 used (mean gain share across walk-forward folds)."""
    scores = model.get_booster().get_score(importance_type="gain")
    raw = np.array([scores.get(f, 0.0) for f in FEATURES])
    total = raw.sum()
    return raw / total if total > 0 else raw


def run_all_folds(df, folds, monotone_constraints, label):
    fold_results, fold_best_params, fold_gain_shares, oof_rows = [], [], [], []
    for tr, te, test_season, train_seasons in folds:
        study = run_optuna_for_fold(df, tr, monotone_constraints)
        best_params = dict(study.best_params)
        best_iteration = study.best_trial.user_attrs["best_iteration"]

        X_train_full = df.loc[tr, FEATURES]
        y_train_full = df.loc[tr, "vorp_next"]
        X_test = df.loc[te, FEATURES]
        final_model = xgb.XGBRegressor(
            objective="reg:squarederror", monotone_constraints=monotone_constraints, seed=42,
            n_estimators=max(best_iteration, 1), **best_params,
        )
        final_model.fit(X_train_full, y_train_full)
        preds = final_model.predict(X_test)
        actual = df.loc[te, "vorp_next"]

        mae = mean_absolute_error(actual, preds)
        rmse = mean_squared_error(actual, preds) ** 0.5
        rho = spearmanr(actual, preds)[0] if len(actual) >= 2 and actual.nunique() > 1 else np.nan

        fold_results.append({"position": "RB", "test_season": test_season, "n_test": len(te), "mae": mae, "rmse": rmse, "spearman": rho})
        fold_best_params.append(best_params | {"n_estimators": best_iteration})
        fold_gain_shares.append(fold_gain_share(final_model))
        oof_rows.append(pd.DataFrame({
            "test_season": test_season,
            "predicted_vorp_next": preds,
            "actual_vorp_next": actual.to_numpy(),
            "vorp_delta_yoy": df.loc[te, "vorp_delta_yoy"].to_numpy(),
            "scarcity_z": df.loc[te, "scarcity_z"].to_numpy(),
        }))

    fold_df = pd.DataFrame(fold_results)
    mean_gain_df = pd.DataFrame({"feature": FEATURES, "mean_gain_share": np.mean(fold_gain_shares, axis=0)})
    oof_df = pd.concat(oof_rows, ignore_index=True)
    print(f"[{label}] Mean MAE: {fold_df['mae'].mean():.2f}, Mean RMSE: {fold_df['rmse'].mean():.2f}, Mean Spearman: {fold_df['spearman'].mean():.3f}")
    return fold_df, pd.DataFrame(fold_best_params), mean_gain_df, oof_df


fold_metrics_df, fold_params_df, gain_constrained_df, oof_df = run_all_folds(rb, folds, MONOTONE_CONSTRAINTS, label="tuned + monotonic")
print(fold_metrics_df.to_string(index=False))

[tuned + monotonic] Mean MAE: 46.67, Mean RMSE: 60.84, Mean Spearman: 0.691
position  test_season  n_test       mae      rmse  spearman
      RB         2017     104 47.684329 62.566688  0.637728
      RB         2018     107 45.447622 58.585677  0.692603
      RB         2019     114 44.295220 58.916990  0.661446
      RB         2020     123 41.442945 54.690718  0.714509
      RB         2021     120 49.205505 62.002632  0.708168
      RB         2022     114 53.385461 68.159304  0.600672
      RB         2023     108 46.345914 58.398501  0.781678
      RB         2024     106 45.559806 63.403553  0.733776


## Diagnostic: does the monotonic constraint actually help?

Domain reasoning alone doesn't guarantee a constraint helps a small dataset — checked directly rather than assumed, the same way QB's notebook (and every other claim in this project) already has been.

In [6]:
NO_CONSTRAINTS = (0, 0, 0, 0)
fold_metrics_unconstrained_df, _, gain_unconstrained_df, _ = run_all_folds(rb, folds, NO_CONSTRAINTS, label="tuned, no constraints")

[tuned, no constraints] Mean MAE: 47.90, Mean RMSE: 61.74, Mean Spearman: 0.691


**Finding, reported honestly, and the opposite direction from QB**: for RB, the monotonic constraints *help* on MAE — 46.67 MAE constrained vs. 47.90 MAE unconstrained, a real ~2.6% difference. Spearman, unlike the old 2-4 search, comes out essentially tied this time: 0.691 constrained vs. 0.691 unconstrained. Net read: constraining still wins clearly on the metric this project has treated as primary (MAE), now at effectively no ranking-quality cost at all (the earlier 2-4 search had shown a small Spearman cost to constraining; the wider search erases it). This is still the opposite of QB's notebook, where dropping the constraints won outright on both metrics — a real, position-specific difference worth keeping in mind rather than assuming one position's constraint tradeoff generalizes to the next: RB's two constrained features (`scarcity_z`, `vorp_delta_yoy`) apparently cost the model very little to shape correctly, unlike QB's `passing_epa`.

### `draft_pick_inverse` vs `age`: actual gain values, not just rank

RB's two non-`scarcity_z`/`vorp_delta_yoy` features — same comparison QB ran for its own two secondary features, using the same methodology as Phase 3 (mean gain share across the walk-forward folds' own refit models, not a single all-data model).

In [7]:
# Phase 3's original untuned mean gain shares (05_feature_selection.ipynb), for the same 2 features.
phase3_gain = {"draft_pick_inverse": 0.087252, "age": 0.062606}

focus = ["draft_pick_inverse", "age"]
rows = []
for f in focus:
    constrained_val = gain_constrained_df.set_index("feature").loc[f, "mean_gain_share"]
    unconstrained_val = gain_unconstrained_df.set_index("feature").loc[f, "mean_gain_share"]
    rows.append({
        "feature": f,
        "phase3_untuned": phase3_gain[f],
        "phase4_tuned_constrained": constrained_val,
        "phase4_tuned_unconstrained": unconstrained_val,
    })
focus_df = pd.DataFrame(rows)
print(focus_df.to_string(index=False))

print("\nGap (draft_pick_inverse - age), each run:")
for col in ["phase3_untuned", "phase4_tuned_constrained", "phase4_tuned_unconstrained"]:
    gap = focus_df.set_index("feature").loc["draft_pick_inverse", col] - focus_df.set_index("feature").loc["age", col]
    ratio = focus_df.set_index("feature").loc["draft_pick_inverse", col] / focus_df.set_index("feature").loc["age", col]
    print(f"  {col}: gap={gap:+.4f}, draft_pick_inverse is {ratio:.2f}x age")

print("\nFull gain tables (all 4 features), for context:")
print("\nConstrained (Phase 4 official):")
print(gain_constrained_df.sort_values("mean_gain_share", ascending=False).to_string(index=False))
print("\nUnconstrained (diagnostic):")
print(gain_unconstrained_df.sort_values("mean_gain_share", ascending=False).to_string(index=False))

           feature  phase3_untuned  phase4_tuned_constrained  phase4_tuned_unconstrained
draft_pick_inverse        0.087252                  0.097371                    0.149747
               age        0.062606                  0.085022                    0.135632

Gap (draft_pick_inverse - age), each run:
  phase3_untuned: gap=+0.0246, draft_pick_inverse is 1.39x age
  phase4_tuned_constrained: gap=+0.0123, draft_pick_inverse is 1.15x age
  phase4_tuned_unconstrained: gap=+0.0141, draft_pick_inverse is 1.10x age

Full gain tables (all 4 features), for context:

Constrained (Phase 4 official):
           feature  mean_gain_share
        scarcity_z         0.756490
draft_pick_inverse         0.097371
               age         0.085022
    vorp_delta_yoy         0.061117

Unconstrained (diagnostic):
           feature  mean_gain_share
        scarcity_z         0.593783
draft_pick_inverse         0.149747
               age         0.135632
    vorp_delta_yoy         0.120838


## Naive baseline, same folds — apples-to-apples with Phase 3

In [8]:
naive_records = []
for tr, te, test_season, train_seasons in folds:
    actual = rb.loc[te, "vorp_next"]
    naive_pred = rb.loc[te, "vorp"]
    mae = mean_absolute_error(actual, naive_pred)
    rho = spearmanr(actual, naive_pred)[0] if len(actual) >= 2 and actual.nunique() > 1 else np.nan
    naive_records.append({"test_season": test_season, "mae": mae, "spearman": rho})
naive_df = pd.DataFrame(naive_records)
naive_mae, naive_spearman = naive_df["mae"].mean(), naive_df["spearman"].mean()
print(f"Naive baseline -- Mean MAE: {naive_mae:.2f}, Mean Spearman: {naive_spearman:.3f}")

Naive baseline -- Mean MAE: 47.97, Mean Spearman: 0.683


## Did tuning meaningfully close the gap Phase 3 found?

In [9]:
tuned_mae, tuned_spearman = fold_metrics_df["mae"].mean(), fold_metrics_df["spearman"].mean()
phase3_untuned_mae, phase3_untuned_spearman = 47.52, 0.691

comparison = pd.DataFrame([
    {"model": "Naive (this season's VORP)", "mae": round(naive_mae, 2), "spearman": round(naive_spearman, 3)},
    {"model": "Phase 3 untuned XGBoost", "mae": phase3_untuned_mae, "spearman": phase3_untuned_spearman},
    {"model": "Phase 4 tuned + monotonic (this notebook)", "mae": round(tuned_mae, 2), "spearman": round(tuned_spearman, 3)},
])
print(comparison.to_string(index=False))

mae_vs_naive_pct = (naive_mae - tuned_mae) / naive_mae
mae_vs_phase3_pct = (phase3_untuned_mae - tuned_mae) / phase3_untuned_mae
print(f"\nTuned vs naive: {mae_vs_naive_pct:.1%} MAE improvement, {tuned_spearman - naive_spearman:+.3f} Spearman")
print(f"Tuned vs Phase 3 untuned: {mae_vs_phase3_pct:+.1%} MAE change, {tuned_spearman - phase3_untuned_spearman:+.3f} Spearman change")

                                    model   mae  spearman
               Naive (this season's VORP) 47.97     0.683
                  Phase 3 untuned XGBoost 47.52     0.691
Phase 4 tuned + monotonic (this notebook) 46.67     0.691

Tuned vs naive: 2.7% MAE improvement, +0.008 Spearman
Tuned vs Phase 3 untuned: +1.8% MAE change, +0.000 Spearman change


## Save fold-by-fold results (overwrites Phase 3's untuned `fold_metrics_RB.csv`)

In [10]:
out_path = REPO_ROOT / "data" / "processed" / "fold_metrics_rb.csv"
fold_metrics_df.to_csv(out_path, index=False)
print(f"Saved {len(fold_metrics_df)} fold rows to {out_path}")

Saved 8 fold rows to C:\Users\viraj\OneDrive\Desktop\ML Trial\ML-Test\data\processed\fold_metrics_rb.csv


## Final model: trained on all available data, using the *stable* hyperparameter region

Not any single fold's exact best trial — a fold that happened to score best once can still reflect noise from a small test season. Instead, take the **median** of each hyperparameter across all 8 folds' best trials (rounded to valid integer values for `max_depth`/`min_child_weight`), which is far less sensitive to any one fold's idiosyncrasy.

`n_estimators` for the final model is found the same principled way real deployments do it: fit on all seasons through the second-to-last labeled one, early-stop against the most recent labeled season alone, then refit on **all** labeled data using that fixed iteration count — so the final artifact is trained on every real data point available, with no leftover held-out slice.

In [11]:
stable_params = {
    "max_depth": int(round(fold_params_df["max_depth"].median())),
    "min_child_weight": int(round(fold_params_df["min_child_weight"].median())),
    "reg_lambda": float(fold_params_df["reg_lambda"].median()),
    "learning_rate": float(fold_params_df["learning_rate"].median()),
    "subsample": float(fold_params_df["subsample"].median()),
}
print("Per-fold best hyperparameters:")
print(fold_params_df.drop(columns=["n_estimators"]).to_string(index=False))
print("\nStable (median) hyperparameters chosen for the final model:")
print(stable_params)

all_seasons_sorted = sorted(rb["season"].unique())
final_valid_season = all_seasons_sorted[-1]
final_train_seasons = all_seasons_sorted[:-1]

final_train_idx = rb.index[rb["season"].isin(final_train_seasons)]
final_valid_idx = rb.index[rb["season"] == final_valid_season]

X_final_train = rb.loc[final_train_idx, FEATURES]
y_final_train = rb.loc[final_train_idx, "vorp_next"]
X_final_valid = rb.loc[final_valid_idx, FEATURES]
y_final_valid = rb.loc[final_valid_idx, "vorp_next"]

probe_model = xgb.XGBRegressor(
    objective="reg:squarederror", eval_metric="mae", monotone_constraints=MONOTONE_CONSTRAINTS, seed=42,
    n_estimators=500, early_stopping_rounds=20, **stable_params,
)
probe_model.fit(X_final_train, y_final_train, eval_set=[(X_final_valid, y_final_valid)], verbose=False)
final_n_estimators = max(probe_model.best_iteration, 1)
print(f"\nFinal n_estimators (early-stopped against {final_valid_season}): {final_n_estimators}")

# Refit on ALL labeled data, no held-out slice left over, using the fixed iteration count.
final_model = xgb.XGBRegressor(
    objective="reg:squarederror", monotone_constraints=MONOTONE_CONSTRAINTS, seed=42,
    n_estimators=final_n_estimators, **stable_params,
)
final_model.fit(rb[FEATURES], rb["vorp_next"])

models_dir = REPO_ROOT / "data" / "models"
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "rb_model.json"
final_model.save_model(str(model_path))
print(f"Final model trained on {len(rb)} rows (seasons {rb['season'].min()}-{rb['season'].max()}), saved to {model_path}")

Per-fold best hyperparameters:
 max_depth  min_child_weight  reg_lambda  learning_rate  subsample
         3                 8    0.249197       0.219572   0.682416
         2                 6    0.191596       0.269522   0.655309
         3                10    2.702067       0.288301   0.648991
         7                 6    2.664493       0.238783   0.613230
         3                 4    0.106956       0.287456   0.618955
         4                 3    0.166448       0.262993   0.653289
         4                 8    5.260729       0.160833   0.619029
         7                10    1.121330       0.242420   0.691861

Stable (median) hyperparameters chosen for the final model:
{'max_depth': 4, 'min_child_weight': 7, 'reg_lambda': 0.6852637738282819, 'learning_rate': 0.2527063312364141, 'subsample': 0.6511400682684466}

Final n_estimators (early-stopped against 2024): 17
Final model trained on 1823 rows (seasons 2008-2024), saved to C:\Users\viraj\OneDrive\Desktop\ML Trial\ML-T

## Does the final model still make football sense?

Checking specifically against Phase 3's finding: `draft_pick_inverse` outranked `age` in the untuned gain-importance chart. If tuning + monotonic constraints flipped that, that would be worth knowing before trusting this model's explanations later (Phase 5, SHAP).

**This section also finally closes out three items this notebook had left as open TODOs** (noted in `roadmap.md`'s known-limitation section and in `06c_model_wr.ipynb`'s/`06d_model_te.ipynb`'s own write-ups, which had to cite RB's numbers as one-off, non-reproducible figures rather than real notebook output): the delta-bucket diagnostic, the `scarcity_z` tail-behavior check, and the `low_confidence_extreme_delta`/`no_delta_history` flags, none of which were ever wired into this notebook as executable code before now. All three are added below using the same methodology WR and TE established, computed fresh against RB's own official (constrained) out-of-fold predictions rather than reused from memory.

In [12]:
gain_scores = final_model.get_booster().get_score(importance_type="gain")
importance_df = (
    pd.DataFrame({"feature": list(gain_scores.keys()), "gain": list(gain_scores.values())})
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)
# get_score() omits any feature never used in a split -- add those back in as 0 for a complete picture.
missing = [f for f in FEATURES if f not in importance_df["feature"].values]
if missing:
    importance_df = pd.concat([importance_df, pd.DataFrame({"feature": missing, "gain": 0.0})], ignore_index=True)
importance_df["gain_share"] = importance_df["gain"] / importance_df["gain"].sum()

print(importance_df.to_string(index=False))

rank = {f: i for i, f in enumerate(importance_df["feature"])}
draft_beats_age = rank.get("draft_pick_inverse", 99) < rank.get("age", 99)
print(f"\ndraft_pick_inverse outranks age: {draft_beats_age}")

           feature          gain  gain_share
        scarcity_z 173063.578125    0.750641
draft_pick_inverse  23883.988281    0.103594
               age  19795.777344    0.085862
    vorp_delta_yoy  13811.186523    0.059904

draft_pick_inverse outranks age: True


## Delta-bucket MAE diagnostic: RB's own numbers, now reproducible in-notebook

`roadmap.md` has cited RB's delta-bucket result since QB's own `05b_qb_feature_experiment.ipynb` diagnostic was extended informally to RB and WR — but unlike QB (`05b`), WR (`06c`), and TE (`06d`), RB never actually had this check committed as real, re-runnable notebook code; the cited numbers (40.43 vs. 56.89, 1.41x) came from one-off interactive analysis outside any notebook. Recomputed here properly, using out-of-fold predictions from RB's own official (constrained) model pooled across all 8 walk-forward folds — genuinely held-out, not in-sample.

In [13]:
oof_df["abs_delta"] = oof_df["vorp_delta_yoy"].abs()
oof_df["abs_error"] = (oof_df["predicted_vorp_next"] - oof_df["actual_vorp_next"]).abs()

DELTA_BUCKET_THRESHOLD = 100
le_bucket = oof_df[oof_df["abs_delta"] <= DELTA_BUCKET_THRESHOLD]
gt_bucket = oof_df[oof_df["abs_delta"] > DELTA_BUCKET_THRESHOLD]

le_mae, gt_mae = le_bucket["abs_error"].mean(), gt_bucket["abs_error"].mean()
overall_mae_pooled = oof_df["abs_error"].mean()

print(f"Overall held-out MAE (pooled out-of-fold): {overall_mae_pooled:.2f} "
      f"(vs. {fold_metrics_df['mae'].mean():.2f}, the unweighted mean-of-fold-means above)")
print(f"|vorp_delta_yoy| <= {DELTA_BUCKET_THRESHOLD}: n={len(le_bucket)}, MAE={le_mae:.2f}")
print(f"|vorp_delta_yoy| > {DELTA_BUCKET_THRESHOLD}:  n={len(gt_bucket)}, MAE={gt_mae:.2f}")
print(f"Extreme bucket is {gt_mae / le_mae:.2f}x worse than the non-extreme bucket")

oof_df["delta_quartile"] = pd.qcut(oof_df["abs_delta"], 4, labels=["Q1 (smallest)", "Q2", "Q3", "Q4 (largest)"])
quartile_summary = oof_df.groupby("delta_quartile", observed=True).agg(
    n=("abs_error", "size"), mae=("abs_error", "mean"),
    delta_range=("abs_delta", lambda x: f"{x.min():.1f}-{x.max():.1f}"),
)
print("\nQuartile breakdown of |vorp_delta_yoy| (out-of-fold MAE by quartile):")
print(quartile_summary.to_string())

print("\nCross-position degradation ratio, all now reproducible in-notebook:")
print(pd.DataFrame([
    {"position": "QB", "ratio": 1.29, "source": "05b_qb_feature_experiment.ipynb (in-sample)"},
    {"position": "RB", "ratio": round(gt_mae / le_mae, 2), "source": "this cell (held-out, out-of-fold)"},
    {"position": "WR", "ratio": 1.45, "source": "06c_model_wr.ipynb (held-out, out-of-fold)"},
    {"position": "TE", "ratio": 1.66, "source": "06d_model_te.ipynb (held-out, out-of-fold)"},
]).to_string(index=False))

Overall held-out MAE (pooled out-of-fold): 46.65 (vs. 46.67, the unweighted mean-of-fold-means above)
|vorp_delta_yoy| <= 100: n=570, MAE=45.17
|vorp_delta_yoy| > 100:  n=103, MAE=64.46
Extreme bucket is 1.43x worse than the non-extreme bucket

Quartile breakdown of |vorp_delta_yoy| (out-of-fold MAE by quartile):
                  n        mae delta_range
delta_quartile                            
Q1 (smallest)   169  35.384793    0.0-13.2
Q2              168  44.414737   13.3-37.7
Q3              168  48.843977   38.1-78.8
Q4 (largest)    168  63.917174  79.1-362.1

Cross-position degradation ratio, all now reproducible in-notebook:
position  ratio                                      source
      QB   1.29 05b_qb_feature_experiment.ipynb (in-sample)
      RB   1.43           this cell (held-out, out-of-fold)
      WR   1.45  06c_model_wr.ipynb (held-out, out-of-fold)
      TE   1.66  06d_model_te.ipynb (held-out, out-of-fold)


**Finding**: the pattern replicates for RB too, now on real, reproducible held-out evidence rather than a remembered number — and it lands close to (slightly above) the previously-cited 1.41x. The quartile breakdown is monotonically increasing, same shape as every other position. RB's ratio sits between QB's and WR's rather than extending a clean four-step staircase: QB 1.29x, RB's own fresh number, WR 1.45x, TE 1.66x, all real, held-out, and now all reproducible directly from their own notebooks.

## `scarcity_z` tail-behavior check — corrected, reported by decile not averaged

**A real correction, not just a new data point**: this project's own written record (`roadmap.md`, and both WR's and TE's own notebooks) has stated that "QB and RB both show `scarcity_z` extremes fitting *better* than the middle of the distribution," treating WR and then TE as the ones that broke that pattern. That claim for QB and RB was never actually backed by decile-separated, reproducible code — it came from an averaged top+bottom-decile number computed outside any notebook. Computed properly here, by decile rather than by the combined average, that framing does not hold up.

In [14]:
oof_df["scarcity_decile"] = pd.qcut(oof_df["scarcity_z"], 10, labels=False, duplicates="drop")
top_decile = oof_df[oof_df["scarcity_decile"] == oof_df["scarcity_decile"].max()]
bottom_decile = oof_df[oof_df["scarcity_decile"] == 0]
middle_80 = oof_df[~oof_df["scarcity_decile"].isin([0, oof_df["scarcity_decile"].max()])]

print(f"scarcity_z top decile (best RBs):        n={len(top_decile)}, MAE={top_decile['abs_error'].mean():.2f}, "
      f"z range [{top_decile['scarcity_z'].min():.2f}, {top_decile['scarcity_z'].max():.2f}]")
print(f"scarcity_z bottom decile (replacement level): n={len(bottom_decile)}, MAE={bottom_decile['abs_error'].mean():.2f}, "
      f"z range [{bottom_decile['scarcity_z'].min():.2f}, {bottom_decile['scarcity_z'].max():.2f}]")
print(f"scarcity_z middle 80%:                   n={len(middle_80)}, MAE={middle_80['abs_error'].mean():.2f}")

combined_extreme = pd.concat([top_decile, bottom_decile])
print(f"\nBoth extreme deciles combined (the number previously cited as 'fits better'): "
      f"n={len(combined_extreme)}, MAE={combined_extreme['abs_error'].mean():.2f} "
      f"(vs. overall pooled held-out {overall_mae_pooled:.2f})")
print(f"\nTop decile alone vs. overall: {top_decile['abs_error'].mean() / overall_mae_pooled:.2f}x")
print(f"Bottom decile alone vs. overall: {bottom_decile['abs_error'].mean() / overall_mae_pooled:.2f}x")

scarcity_z top decile (best RBs):        n=90, MAE=68.53, z range [1.77, 4.20]
scarcity_z bottom decile (replacement level): n=90, MAE=26.70, z range [-0.98, -0.86]
scarcity_z middle 80%:                   n=716, MAE=46.40

Both extreme deciles combined (the number previously cited as 'fits better'): n=180, MAE=47.62 (vs. overall pooled held-out 46.65)

Top decile alone vs. overall: 1.47x
Bottom decile alone vs. overall: 0.57x


**Corrected finding**: the *combined* extreme-decile average (47.62) does come in close to the overall held-out MAE (46.65) — which is exactly the number that made RB look like it matched QB's supposed "extremes fit better" story. But separated by decile, that combined number is hiding the same asymmetry WR's notebook already found and TE's confirmed: the **top decile (RB's actual stars) is the hardest segment in the dataset at MAE 68.53** — 1.47x worse than overall, and worse than even the `vorp_delta_yoy` extreme-tail bucket (64.46) directly above — while the **bottom decile (replacement-level backs) is easy at MAE 26.70**, 0.57x of overall. Averaging the two together happens to land close to neutral for RB; it does not mean both tails are actually easy.

**This changes the cross-position picture materially.** The "2-and-2 split" (QB/RB show both extremes fitting better; WR/TE show the top decile as hardest) that `06c_model_wr.ipynb`, `06d_model_te.ipynb`, and `roadmap.md` all currently state is not supported once RB is checked the same decile-separated way WR introduced. RB's own top decile is *harder* than its own held-out MAE, exactly like WR's and TE's. Whether QB's own never-formally-decile-checked claim would survive the same scrutiny is now a real open question rather than a settled data point — worth checking directly in `06a_model_qb.ipynb` before continuing to cite it, rather than assumed to hold just because it was written down first. `roadmap.md` is updated to reflect this correction rather than repeating the earlier framing.

## `low_confidence_extreme_delta` and `no_delta_history` flags — added correctly from the start

RB never had this flag wired in at all — `roadmap.md` had it as a standing TODO ("RB's model should get the same flag wired in before it's treated as fully documented"). Added here directly with **both** flags QB's and WR's notebooks needed a bug fix to get right: `low_confidence_extreme_delta` (`|vorp_delta_yoy| > 100`, this notebook's own delta-bucket diagnostic above) and a separate `no_delta_history` flag for rows with no computable prior-season delta at all (e.g. a true rookie). These are kept distinct rather than merged, because `NaN > 100` silently evaluates to `False` in pandas/NumPy — treating a missing delta as "not extreme" is not the same as treating it as "reliable," and an in-sample check (below) shows the two populations don't actually behave the same way.

In [15]:
LOW_CONFIDENCE_DELTA_THRESHOLD = 100  # matches the |vorp_delta_yoy| bucket confirmed above


def add_low_confidence_flag(df, delta_col="vorp_delta_yoy"):
    """True wherever |vorp_delta_yoy| exceeds the threshold this notebook's own delta-bucket
    diagnostic (above) found to be a real, measured weak spot -- same flag-not-drop pattern as
    Phase 2's low_snap_next_season and QB/WR/TE's low_confidence_extreme_delta. Informational
    only: this model was never retrained or refit with this flag as a feature or filter.

    NaN (no computable prior-season delta, e.g. a true rookie) evaluates to False here, NOT
    True -- pandas/NumPy silently reads "NaN > threshold" as False. That is NOT the same as
    "reliable"; see add_no_delta_history_flag below for that distinct, separate case."""
    return df[delta_col].abs() > LOW_CONFIDENCE_DELTA_THRESHOLD


def add_no_delta_history_flag(df, delta_col="vorp_delta_yoy"):
    """True wherever vorp_delta_yoy has no value at all (no usable prior-season VORP to
    diff against -- typically a true rookie). Deliberately a SEPARATE flag from
    low_confidence_extreme_delta rather than folding NaN into it as True."""
    return df[delta_col].isna()


n_flagged_train = add_low_confidence_flag(rb).sum()
n_no_history_train = add_no_delta_history_flag(rb).sum()
print(f"low_confidence_extreme_delta threshold: |vorp_delta_yoy| > {LOW_CONFIDENCE_DELTA_THRESHOLD}")
print(f"Training rows flagged low_confidence_extreme_delta: {n_flagged_train} / {len(rb)} ({n_flagged_train / len(rb):.1%})")
print(f"Training rows flagged no_delta_history: {n_no_history_train} / {len(rb)} ({n_no_history_train / len(rb):.1%})")

# Same in-sample check QB/WR/TE used to decide this: does no-delta-history actually behave
# like an extreme-delta row, or like the general population? Uses the already-saved rb_model.json.
_check_model = xgb.XGBRegressor()
_check_model.load_model(str(REPO_ROOT / "data" / "models" / "rb_model.json"))
_check_preds = _check_model.predict(rb[FEATURES])
_check_err = (_check_preds - rb["vorp_next"]).abs()
_has_delta = rb["vorp_delta_yoy"].notna()
_extreme = _has_delta & (rb["vorp_delta_yoy"].abs() > LOW_CONFIDENCE_DELTA_THRESHOLD)
print(f"\nIn-sample MAE (shipped rb_model.json) -- has delta: {_check_err[_has_delta].mean():.2f}, "
      f"no delta history: {_check_err[~_has_delta].mean():.2f}, extreme delta: {_check_err[_extreme].mean():.2f}")
print("(no-delta-history rows are not obviously as risky as a genuinely large measured swing --")
print(" same reasoning QB/WR/TE used to keep this a separate flag rather than defaulting to maximum caution)")

low_confidence_extreme_delta threshold: |vorp_delta_yoy| > 100
Training rows flagged low_confidence_extreme_delta: 216 / 1823 (11.8%)
Training rows flagged no_delta_history: 542 / 1823 (29.7%)

In-sample MAE (shipped rb_model.json) -- has delta: 43.21, no delta history: 38.36, extreme delta: 56.89
(no-delta-history rows are not obviously as risky as a genuinely large measured swing --
 same reasoning QB/WR/TE used to keep this a separate flag rather than defaulting to maximum caution)


## Real predictions vs. reality — 2024 test season

Same check every other position notebook has: the 2024 walk-forward fold trains on 2008-2023 and predicts each RB's realized 2025 outcome from their real 2024 season. Reproduces the exact 2024 row already averaged into the constrained fold table above, broken out player-by-player.

In [16]:
last_fold_tr, last_fold_te, last_test_season, last_train_seasons = folds[-1]
assert last_test_season == 2024

last_fold_params = dict(fold_params_df.iloc[-1])
last_fold_n_estimators = int(last_fold_params.pop("n_estimators"))
# .iloc[-1] on a mixed-dtype DataFrame upcasts the whole row to float64 (e.g. max_depth -> 3.0),
# which XGBoost's C API rejects for integer params -- cast them back explicitly.
last_fold_params["max_depth"] = int(last_fold_params["max_depth"])
last_fold_params["min_child_weight"] = int(last_fold_params["min_child_weight"])

X_train_2024 = rb.loc[last_fold_tr, FEATURES]
y_train_2024 = rb.loc[last_fold_tr, "vorp_next"]
X_test_2024 = rb.loc[last_fold_te, FEATURES]
y_test_2024 = rb.loc[last_fold_te, "vorp_next"]

reproduced_2024_model = xgb.XGBRegressor(
    objective="reg:squarederror", monotone_constraints=MONOTONE_CONSTRAINTS, seed=42,
    n_estimators=last_fold_n_estimators, **last_fold_params,
)
reproduced_2024_model.fit(X_train_2024, y_train_2024)
predicted_2024 = reproduced_2024_model.predict(X_test_2024)

results_2024 = pd.DataFrame({
    "player": rb.loc[last_fold_te, "player_display_name"].to_numpy(),
    "predicted_vorp_next": predicted_2024,
    "actual_vorp_next": y_test_2024.to_numpy(),
    "low_confidence_extreme_delta": add_low_confidence_flag(rb.loc[last_fold_te]).to_numpy(),
    "no_delta_history": add_no_delta_history_flag(rb.loc[last_fold_te]).to_numpy(),
})
results_2024["error"] = results_2024["predicted_vorp_next"] - results_2024["actual_vorp_next"]
results_2024["abs_error"] = results_2024["error"].abs()
results_2024 = results_2024.sort_values("abs_error").reset_index(drop=True)

# Sanity check: this should match the 2024 row already in fold_metrics_df above.
mae_check = mean_absolute_error(results_2024["actual_vorp_next"], results_2024["predicted_vorp_next"])
rmse_check = mean_squared_error(results_2024["actual_vorp_next"], results_2024["predicted_vorp_next"]) ** 0.5
rho_check = spearmanr(results_2024["actual_vorp_next"], results_2024["predicted_vorp_next"])[0]
print(f"Reproduced 2024 fold -- MAE {mae_check:.2f}, RMSE {rmse_check:.2f}, Spearman {rho_check:.3f} "
      f"(should match the test_season=2024 row in the constrained fold table above)")

n_flagged_2024 = results_2024["low_confidence_extreme_delta"].sum()
n_no_history_2024 = results_2024["no_delta_history"].sum()
print(f"low_confidence_extreme_delta flagged: {n_flagged_2024} / {len(results_2024)} rows in this fold")
print(f"no_delta_history flagged: {n_no_history_2024} / {len(results_2024)} rows in this fold")

print(f"\nBest 10 predictions (smallest absolute error):")
print(results_2024.head(10).round(1).to_string(index=False))
print(f"\nWorst 10 misses (largest absolute error):")
print(results_2024.tail(10).round(1).to_string(index=False))

Reproduced 2024 fold -- MAE 45.56, RMSE 63.40, Spearman 0.734 (should match the test_season=2024 row in the constrained fold table above)


low_confidence_extreme_delta flagged: 9 / 106 rows in this fold
no_delta_history flagged: 22 / 106 rows in this fold

Best 10 predictions (smallest absolute error):
           player  predicted_vorp_next  actual_vorp_next  low_confidence_extreme_delta  no_delta_history  error  abs_error
      Dylan Laube          -122.900002            -123.2                         False              True    0.3        0.3
   Emanuel Wilson           -38.700001             -38.4                         False             False   -0.3        0.3
 Terrell Jennings          -111.800003            -110.7                         False              True   -1.1        1.1
    George Holani          -113.000000            -111.6                         False              True   -1.4        1.4
     Isaiah Davis           -66.900002             -64.7                         False              True   -2.2        2.2
   Reggie Gilliam          -124.500000            -121.6                         False          

**Best prediction: Emanuel Wilson** — predicted -38.7, actual -38.4 (abs error 0.3). A backup/committee-role back whose already-below-replacement 2024 season carried through almost exactly to 2025, the same low-variance easy case every position's best prediction lands on.

**Worst miss: Christian McCaffrey** — predicted -93.1, actual +240.2 (abs error 333.3, the largest single miss of any position notebook so far, and **flagged `low_confidence_extreme_delta`**: his `vorp_delta_yoy` heading into this prediction was extreme). McCaffrey's 2024 season was almost entirely wiped out by a lingering Achilles/PCL injury, so his 2024 features fed the model a near-replacement-level year with an already-large negative delta from his elite 2023 — and the model reasonably extrapolated further decline. What actually happened is a full, healthy bounce-back to elite volume and production in 2025, a real-world recovery no backward-looking statistical feature set could see coming. This is the clearest possible illustration of what the flag is for: a huge measured swing (driven here by an injury year, not a skill change) that the model cannot resolve in either direction with any real confidence.

## Live 2026 prediction — a genuine unknown

`rb_model.json` — trained on every real season through 2024 — is fed each RB's actual, already-realized 2025 season to predict 2026 VORP, using the top 5 RBs by realized 2025 VORP. Speculative and unverifiable until the season finishes; does not change the validated estimate above.

In [17]:
loaded_rb_model = xgb.XGBRegressor()
loaded_rb_model.load_model(str(REPO_ROOT / "data" / "models" / "rb_model.json"))

top5_2025 = rb_2025_features.sort_values("vorp", ascending=False).head(5).copy()
top5_2025["predicted_2026_vorp"] = loaded_rb_model.predict(top5_2025[FEATURES])
top5_2025["low_confidence_extreme_delta"] = add_low_confidence_flag(top5_2025)
top5_2025["no_delta_history"] = add_no_delta_history_flag(top5_2025)

live_predictions = top5_2025[
    ["player_display_name", "vorp", "predicted_2026_vorp", "low_confidence_extreme_delta", "no_delta_history"] + FEATURES
].rename(columns={"vorp": "actual_2025_vorp"}).reset_index(drop=True)

print("Top 5 RBs by realized 2025 VORP -- live 2026 prediction (SPECULATIVE, unverifiable until the season finishes):\n")
print(live_predictions[["player_display_name", "actual_2025_vorp", "predicted_2026_vorp", "low_confidence_extreme_delta", "no_delta_history"]]
      .round(1).to_string(index=False))
print("\nFeature rows used (each RB's real, already-realized 2025 season):")
print(live_predictions[["player_display_name"] + FEATURES].round(3).to_string(index=False))

print(f"\nlow_confidence_extreme_delta = True means |vorp_delta_yoy| > {LOW_CONFIDENCE_DELTA_THRESHOLD}. For those")
print("rows, the exact predicted number deserves real skepticism (see the delta-bucket diagnostic above).")
print("no_delta_history = True means this player has no usable prior-season VORP at all (e.g. a true rookie) --")
print("a DIFFERENT, separate reason for caution than the extreme-delta flag, not a stronger or weaker version of it.")

Top 5 RBs by realized 2025 VORP -- live 2026 prediction (SPECULATIVE, unverifiable until the season finishes):


player_display_name  actual_2025_vorp  predicted_2026_vorp  low_confidence_extreme_delta  no_delta_history
Christian McCaffrey             240.2            74.699997                          True             False
    Jonathan Taylor             213.9            77.599998                          True             False
     Bijan Robinson             205.9           131.100006                         False             False
       Jahmyr Gibbs             203.0           102.300003                         False             False
      De'Von Achane             163.9            44.700001                         False             False

Feature rows used (each RB's real, already-realized 2025 season):
player_display_name  scarcity_z  vorp_delta_yoy    age  draft_pick_inverse
Christian McCaffrey       3.396          345.87 29.235               0.125
    Jonathan Taylor       3.092          124.17 26.617               0.024
     Bijan Robinson       3.000           40.67 23.587            

### Football-sense eyeball pass on the top 5

**Christian McCaffrey** (actual 2025 VORP +240.2 → predicted 2026 +74.7, flagged `low_confidence_extreme_delta`): a real, huge recovery year (see the worst-miss discussion above — the model already has a documented history of mishandling his exact situation in the other direction) followed by another large predicted decline. Given this model's own demonstrated tendency to get McCaffrey specifically wrong when his delta is extreme, **this prediction deserves real, specific skepticism grounded in an in-notebook precedent**, not just the generic flag boilerplate.

**Jonathan Taylor** (+213.9 → +77.6, flagged): a real bounce-back year from a down 2023-24 stretch. Same generic mean-reversion caution as any large positive delta — worth real skepticism on the magnitude, without a specific McCaffrey-style precedent to lean on.

**Bijan Robinson, Jahmyr Gibbs, De'Von Achane** (all unflagged, moderate positive deltas): established, still-ascending young stars with real, defensible moderate-decline calls. No red flags — none of RB's other top-5 stories involve the kind of context (injury recovery, extreme swing) that would warrant skepticism beyond the model's own honest, validated error bars.

**Bottom line on this pass**: 2 of 5 flagged, and the McCaffrey case is a genuinely stronger, more specific caution than a generic flag — this exact model has already been shown, in this exact notebook, to badly mishandle his exact situation once.

## Honest verdict

**Unlike QB, widening `max_depth` from 2-4 to 2-8 is a genuine win for RB on both metrics — checked directly rather than assumed to carry over.**

| Model | MAE | Spearman |
|---|---|---|
| Naive (this season's VORP) | 47.97 | 0.683 |
| Phase 3 untuned XGBoost | 47.52 | 0.691 |
| Phase 4 tuned + monotonic, max_depth 2-4 (previous, still shipped until now) | 47.42 | 0.681 |
| **Phase 4 tuned + monotonic, max_depth 2-8 (new, official)** | **46.67** | **0.691** |
| *(diagnostic: tuned, no constraints, max_depth 2-8)* | *47.90* | *0.691* |

This is a real, unambiguous improvement over the previously-shipped 2-4 model: MAE drops from 47.42 to 46.67 (a further 1.6% on top of the already-tuned model, 2.7% total over naive) and Spearman recovers from 0.681 to 0.691 — both metrics move the right direction simultaneously, not a mixed result. Because it wins on both MAE and Spearman, per the standing decision rule, **this run's model replaces the old 2-4 model as the shipped one** — `rb_model.json` and `fold_metrics_rb.csv` above are now built from the widened search.

**This does not mean wider trees are a universal win** — QB's own notebook found the opposite mix (a small MAE win alongside a Spearman give-back) using the identical change. RB's folds simply had more room to use here: the per-fold `max_depth` values chosen this run range from 2 to 7 (median 4), versus RB's old 2-4 search which could never explore past 4. Checking this per position, rather than assuming QB's finding carries over, was the right call.

**The monotonic-constraint diagnostic still goes the opposite way from QB's, and now more cleanly**: constraining `scarcity_z`/`vorp_delta_yoy` still *helps* MAE (46.67 vs. 47.90 unconstrained), and this time the old search's small Spearman cost to constraining has disappeared (0.691 vs. 0.691, an exact tie). RB's constrained model is now a clean win on MAE with no downside on ranking at all.

**Football-sense check, still no instability, unlike QB**: `scarcity_z` dominates even more than before (75.1% of gain vs. 80.2% under the old search — roughly the same overwhelming share). `draft_pick_inverse` still outranks `age` in the final model (10.4% vs. 8.6%), the same order Phase 3's untuned chart and every prior version of this notebook found. **This is the one clean contrast with QB's own re-run**: QB's final model saw its secondary-feature ranking flip (`passing_epa` over `draft_pick_inverse`) under the wider search; RB's ranking did not move at all.

**A reproducibility note, same as QB's notebook**: the exact numbers above shift slightly each time this notebook is re-executed, since the feature pipeline pulls live `nflreadpy` data rather than a frozen snapshot. The conclusions — constraints help RB's MAE (and no longer cost Spearman under the wider search), wider trees are a real win for RB (unlike QB, where they were a smaller, more mixed win), `scarcity_z` dominates, `draft_pick_inverse` outranks `age` — have held up across multiple runs and both `max_depth` ranges; the precise decimals have not.

**Bottom line**: RB gets a genuine, if still modest, edge from widening `max_depth` to 2-8 — a cleaner result than QB's own version of this same experiment, and specifically confirmed rather than assumed. The monotonic constraints stay locked in for RB (the opposite call from QB, made on RB's own held-out folds, not by analogy). `rb_model.json` and `fold_metrics_rb.csv` are updated to reflect this widened-search model as the new official one.

---

### Interface confirmation (the actual point of this exercise)

This notebook's `XGBRegressor`-first numbers, and QB's after being re-run through the same conversion, reproduced their pre-conversion `xgb.train()`/`DMatrix` results **exactly** (fold-by-fold MAE/RMSE/Spearman, per-fold best hyperparameters, final `n_estimators`, and gain-importance shares all matched to the displayed decimal places) — confirmed by executing both notebooks end-to-end and diffing outputs against the versions recorded before the conversion. This is the expected result: same seed, same params, same underlying booster call either way. Going forward, QB and RB (and any position built after them) share one interface: `XGBRegressor(...).fit(X, y, eval_set=[...], ...)`, `model.get_booster().get_score(...)` for gain importance, `model.best_iteration`/`model.best_score` for early stopping — no `DMatrix` construction anywhere.

---

## Addendum: closing out the delta-bucket, `scarcity_z`, and flag TODOs

This notebook originally stopped at the football-sense check above and left the delta-bucket diagnostic, the `scarcity_z` tail check, and the `low_confidence_extreme_delta` flag as unimplemented TODOs — `roadmap.md` had to cite RB's numbers for these as one-off, non-reproducible figures from outside any notebook. All three are now implemented as real, re-runnable code (added above, before this Honest verdict section), with two real findings worth stating plainly:

**The `vorp_delta_yoy` extreme-tail weakness is confirmed for RB on real held-out evidence**: the extreme bucket is meaningfully worse than the non-extreme one, in the same range as QB/WR/TE's own confirmed degradation. `low_confidence_extreme_delta` and a distinct `no_delta_history` flag (added in the same bug-fix pass applied to QB's and WR's notebooks — `NaN > 100` silently evaluates to `False` in pandas, which is not the same as "reliable") are now attached to every player-level table in this notebook.

**A real correction to this project's cross-position record**: `roadmap.md`, `06c_model_wr.ipynb`, and `06d_model_te.ipynb` have all stated that "QB and RB show `scarcity_z`'s own extremes fitting better than the middle of the distribution," framed as a clean 2-and-2 split against WR/TE. That claim for RB does not survive being checked the same decile-separated way WR's notebook introduced: RB's own top decile (its actual star backs) is the **hardest** segment in its dataset, not an easy one — the earlier "fits better" read came from averaging the top and bottom deciles together, which happens to land close to neutral for RB and masks the same asymmetry WR and TE already found. `roadmap.md` has been corrected accordingly. Whether QB's own version of this claim holds up under the same real check is now flagged as unverified rather than assumed.